# Progetto Architetture Dati: Impatto della Data Quality sui Modelli di Machine Learning
---
---

## 1. Obiettivo del Progetto di Architetture Dati

In ambito aziendale e nelle moderne architetture dati, raramente i modelli di Machine Learning in produzione vengono alimentati con dati perfetti. Sensori malfunzionanti, errori di join, bug nelle API o inserimenti manuali errati sporcano costantemente il flusso informativo.

L'obiettivo di questo progetto è **certificare la robustezza della nostra Rete Neurale** misurando empiricamente e statisticamente il degrado delle sue performance al variare della Data Quality. Il lavoro si articola nei seguenti 5 step:
### **1.1 Step Operativi**
Il lavoro è stato suddiviso in 5 fasi logiche:
1. **Preparazione e Baseline Analysis:** Isolamento della MLP `(128, 128, 64, 32)` e calcolo dell'accuratezza su dati puliti.
2. **Architettura del Degrado e Profilazione:** Sviluppo di moduli per iniettare gli errori nel Train Set.
3. **Automazione degli Esperimenti:** Implementazione di cicli di training iterativi per garantire la validità statistica.
4. **Analisi Correlativa e Visualizzazione:** Studio della relazione tra metriche di qualità e calo della *Test Accuracy*.
5. **Conclusioni Finali:** Valutazione della tolleranza ai guasti del modello.

### **1.2 Architettura del Repository**
il progetto è strutturato come segue:

```text
Architetture-Dati-Progetto/
├── data/
│   └── dataset_ml_ready.csv       # Dataset originale 
├── docs/
│   └── Architetture-Dati report.ipynb      # Questo documento 
├── src/
│   └──  baseline_data_quality.py   # training e valutazione baseline
│   
├── requirements.txt               # Dipendenze 
└── README.md                      # Documentazione tecnica per l'esecuzione


---
---

## 2. Introduzione e Background: ATP Tennis Prediction

Il punto di partenza di questo elaborato è un progetto di Machine Learning precedentemente sviluppato, finalizzato alla previsione dell'esito dei match di tennis ATP dal 2000 al 2024. 

Nel progetto originale, abbiamo costruito una complessa pipeline di Data Preparation per estrarre e calcolare oltre 30 feature storiche e prestazionali per ogni giocatore. Tra le metriche più importanti figurano:
- **Differenziale ELO** globale e per superficie.
- **Forma Recente** misurata in percentuale di vittorie nelle ultime 5, 25 e 50 partite.
- **Metriche di Efficienza** rappresentate dalla percentuale di prime di servizio vincenti, palle break salvate.
- **Fattori Fisici** come età, altezza, mano dominantee.

Il modello definitivo scelto è stato una **Rete Neurale Multilayer Perceptron** con architettura `(128, 128, 64, 32)` e regolarizzazione `alpha=0.01`. Addestrata sui dati scalati dal 2000 al 2022, la rete ha raggiunto un'accuratezza solida e stabile (tra il 65% e il 71% a seconda delle configurazioni) sul Test Set comprendente tutte le partite degli anni 2023 e 2024. Infine è stato fatta una validazione finale calcolando l'accuracy del nostro modello simulando il torneo degli Australian Open 2025 e ottenendo gli stessi valori per le metriche riscontrate sul Test Set. 

Per maggiori dettagli sul modello utilizzato è stato incluso il report del progetto in doc/tennis-prediction report.ipynb e l'intero progetto è consultabile al link:  ---**INSERIAMO IL LINK DEL GITHUB** ---

---
---

# 3. Preparazione e Baseline Analysis

In questo primo stadio dell'esperimento, prepariamo l'infrastruttura tecnica e calcoliamo la **Baseline**. La Baseline rappresenta l'accuratezza del modello su dati "puliti" e servirà come termine di paragone per misurare il degrado delle performance nelle fasi successive. Al fine di una misurazione efficiente sono state estratte le sezioni di codice utili al nostro progetto dal progetto tennis-prediction e sono state racchiuse in metodi da poter chiamare all'occorrenza per gli allenamenti che verranno fatti durante ogni esperimento, e sono stati inseriti nello script baseline_data_quality.py.

In questo primo step operativo, procediamo con l'importazione delle librerie fondamentali per la manipolazione dei dati (`pandas`, `numpy`) e per la costruzione della pipeline di Machine Learning tramite *scikit-learn* (`MLPClassifier`, `StandardScaler`). A livello architetturale, abbiamo definito una costante globale denominata `FEATURES` che contiene l'elenco esatto delle sole 34 variabili in input alla rete neurale che verranno usate per l'allenamento.

In [1]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings

warnings.filterwarnings('ignore')

# COSTANTI E FEATURES, usate per l'allenamento e la valutazione del modello
FEATURES = [
    'best_of', 'round_num', 'surface_Hard', 'surface_Clay', 'surface_Grass',
    'diff_age', 'diff_ht', 'p1_is_left', 'p2_is_left',
    'DIFF_ELO', 'DIFF_SURF_ELO', 'ATP_RANK_DIFF', 'DIFF_H2H', 'DIFF_N_GAMES',
    'WIN_LAST_5_DIFF', 'WIN_LAST_25_DIFF', 'WIN_LAST_50_DIFF', 'WIN_LAST_100_DIFF',
    'ELO_GRAD_20_DIFF', 'ELO_GRAD_35_DIFF', 'ELO_GRAD_50_DIFF', 'ELO_GRAD_100_DIFF',
    'ACE_L5_DIFF', 'ACE_L20_DIFF', 'ACE_L50_DIFF',
    'DF_L5_DIFF', 'DF_L20_DIFF', 'DF_L50_DIFF',
    '1ST_WIN_PCT_L5_DIFF', '1ST_WIN_PCT_L20_DIFF', '1ST_WIN_PCT_L50_DIFF',
    'BP_SAVE_PCT_L5_DIFF', 'BP_SAVE_PCT_L20_DIFF', 'BP_SAVE_PCT_L50_DIFF'
]

In questa sezione definiamo due funzioni modulari per garantirne la riusabilità durante i loop di test.

1. **`get_train_test_split`:**
   Questa funzione si occupa di caricare il dataset e separare i dati di addestramento da quelli di test. A livello metodologico, abbiamo optato per uno Split Temporale: usiamo i dati storici fino al 31/12/2022 per il training e blindiamo le stagioni 2023-2024 per il test. Questo approccio è fondamentale nei dati sportivi per evitare che il modello impari dal "futuro". Inoltre, gestisce fisiologici valori mancanti iniziali (`fillna(0)`).

2. **`train_and_evaluate_mlp`:**
   Questa funzione si occupa dell'addestramento del modello. Riceve i dati, applica la standardizzazione e addestra la nostra rete neurale di riferimento (`MLPClassifier` con architettura 128-128-64-32). 
   *Nota Architetturale:* È fondamentale osservare che lo `StandardScaler` viene "fittato" (`fit_transform`) **esclusivamente sul Training Set**. Il Test Set viene solo trasformato (`transform`). Questa accortezza previene **Data Leakage**, garantendo che la valutazione del modello sia non influenzata dai risultati.

In [2]:
# FUNZIONI DI SUPPORTO
# Divisione dei dati per Train e Test basata sulla data del torneo
def get_train_test_split(file_path):
    """Carica il CSV e divide in Train e Test."""
    df = pd.read_csv(file_path)
    df['tourney_date'] = pd.to_datetime(df['tourney_date'])
    
    train_mask = df['tourney_date'] < '2023-01-01'
    test_mask = df['tourney_date'] >= '2023-01-01'
    
    X_train = df.loc[train_mask, FEATURES].fillna(0)
    y_train = df.loc[train_mask, 'target']
    X_test = df.loc[test_mask, FEATURES].fillna(0)
    y_test = df.loc[test_mask, 'target']
    
    return X_train, y_train, X_test, y_test

# Funzione per addestrare la MLP e valutare l'accuracy
def train_and_evaluate_mlp(X_train, y_train, X_test, y_test):
    """Scala i dati, addestra la MLP e ritorna l'accuracy."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 128, 64, 32),
        activation='relu',
        alpha=0.01, 
        max_iter=150, 
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=42
    )
    mlp.fit(X_train_scaled, y_train)
    
    y_pred = mlp.predict(X_test_scaled)
    return accuracy_score(y_test, y_pred)

All'interno di questo blocco andiamo ad eseguire il Training e il Testing del nostro modello, l'addestramento viene eseguito sui dati "puliti" e i log a schermo confermano il corretto partizionamento dei record. Il valore di `baseline_acc` ottenuto in questo passaggio diventerà la nostra "Baseline": il tetto massimo di performance rispetto al quale misureremo e valuteremo tutti i futuri degradi causati dalla scarsa Data Quality.

In [3]:
# ESECUZIONE PRINCIPALE

if __name__ == "__main__":
    print("--- INIZIO PIPELINE DATA QUALITY ---")
    
    # Assicurati che il percorso del file sia corretto!
    dataset_path = "../data/dataset_ml_ready.csv" 
    
    try:
        print("Caricamento dati e split temporale...")
        X_train_clean, y_train_clean, X_test_clean, y_test_clean = get_train_test_split(dataset_path)
        
        print(f"   Train set: {len(X_train_clean)} match | Test set: {len(X_test_clean)} match")
        
        print("Addestramento Rete Neurale Baseline...")
        baseline_acc = train_and_evaluate_mlp(X_train_clean, y_train_clean, X_test_clean, y_test_clean)
        
        print(f"\nAccuracy Baseline = {baseline_acc:.4f}")
        
        
    except FileNotFoundError:
        print(f"\nERRORE: Non trovo il file {dataset_path}")

--- INIZIO PIPELINE DATA QUALITY ---
Caricamento dati e split temporale...
   Train set: 68846 match | Test set: 6009 match
Addestramento Rete Neurale Baseline...

Accuracy Baseline = 0.6572
